# Глава 12. Интеллектуальная поддержка клиентов

## 12.1. Поддержка пользователей на основе RAG

RAG-система поддержки пользователей — это новый подход к клиентскому сервису. Вместо статичных FAQ она обрабатывает каждый запрос с учётом контекста корпоративных знаний и особенностей клиента.

Главные преимущества:

- **Точные ответы** на основе проверенных источников — документации, истории обращений, технических спецификаций.
- **Персонализация** — система адаптирует стиль и глубину ответа под уровень пользователя, его роль и опыт.
- **Скорость** — ответы за секунды благодаря векторному поиску; время решения запросов сокращается примерно на 30%.
- **Сложные запросы** — Multi-Head RAG анализирует разные аспекты вопроса и объединяет информацию из нескольких источников.
- **Доступ к экспертным знаниям** — операторы первой линии решают больше вопросов без эскалации.
- **Многоязычность** — поддержка на родном языке пользователя с учётом культурных и региональных особенностей.
- **Масштабируемость** — тысячи одновременных запросов без роста штата поддержки.
- **Аналитика** — система выявляет частые проблемы и помогает улучшать документацию и продукт.

Итог: RAG делает поддержку быстрее, точнее и персональнее, снижая нагрузку на персонал и улучшая пользовательский опыт.

## 12.2. Чат‑боты нового поколения

Чат-боты нового поколения на базе RAG — это уже не просто поисковики, а умные ассистенты, которые ведут осмысленный диалог, подстраиваются под пользователя и проверяют свои ответы.

**Ключевые принципы их работы:**

1. **Поэтапная обработка запроса** — система формулирует несколько уточняющих запросов, ищет информацию в разных источниках, ранжирует документы и только потом отвечает.

2. **Работа с любыми форматами** — текст, изображения, PDF, аудио, данные дополненной реальности.

3. **Персонализация** — бот запоминает стиль общения и предпочтения пользователя, подстраивая тон и глубину ответа.

4. **Самопроверка** — после генерации ответа бот сверяет его с исходными документами и снижает уверенность при расхождениях.

5. **Дозапрос контекста** — если данных не хватает, система запрашивает дополнительную информацию без заметной задержки.

6. **Объединение источников** — документация, вики, форумы, прошлые обращения — всё используется для решения сложных проблем.

7. **Обучение на обратной связи** — оценки пользователей корректируют ранжирование источников, улучшая качество ответов.

8. **Защита данных** — фильтрация чувствительной информации до передачи в модель, что позволяет работать в regulated-средах.

**Итог:** Сочетание семантического поиска, персонализации, самоконтроля и безопасности выводит клиентский сервис на новый уровень эффективности и доверия.

## 12.3. Интеграция с CRM и другими системами

RAG-чат-боты нужно подключать к системам компании (CRM, ERP, базы знаний и др.), чтобы они давали точные и персонализированные ответы на основе реальных данных.

**Суть по пунктам:**

1. **CRM.** Бот видит историю клиента, его заказы и предпочтения — и отвечает с учётом этого.

2. **Обмен данными.** Бот не только читает данные из CRM, но и сам записывает туда новые обращения и обновляет статусы. Ничего не дублируется.

3. **ERP.** Бот может сказать клиенту про наличие товара, сроки поставки, статус платежа — без участия оператора.

4. **Базы знаний.** Бот ищет ответы в документах компании (инструкции, регламенты, политики) и автоматически учитывает их обновления.

5. **API.** Бота легко встроить в сайт, приложение или внутреннюю систему компании через стандартные интерфейсы.

6. **Тикеты.** Бот анализирует новое обращение, находит похожие решённые случаи и предлагает готовое решение — поддержка работает быстрее.

7. **Все каналы.** Один бот обслуживает чат на сайте, почту, соцсети и приложение — ответы одинаково качественные везде.

8. **Аналитика.** Все вопросы клиентов собираются и показывают, что им непонятно и что нужно улучшить.

9. **Безопасность.** Бот показывает только те данные, к которым у пользователя есть доступ. Все действия записываются.

10. **Будущее.** RAG становится центром корпоративного ИИ — единой точкой доступа ко всем знаниям компании.

## 12.4. Полный код чат‑бота с RAG

Представленная в настоящем подразделе реализация демонстрирует полнофункциональный чат‑бот с поддержкой RAG, построенный на современных технологиях и готовый к корпоративному использованию. Система включает все необходимые компоненты для создания интеллектуального ассистента с доступом к корпоративным базам знаний.

### Основной код чат‑бота

In [ ]:
import os
import json
import logging
from datetime import datetime
from typing import List, Dict, Any, Optional
import asyncio
from dataclasses import dataclass
import streamlit as st
import chromadb
from sentence_transformers import SentenceTransformer
import openai
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.document_loaders import TextLoader, PyPDFLoader, DirectoryLoader
import tiktoken
import requests

# Настройка логирования
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

@dataclass
class ChatMessage:
    """Структура сообщения в чате"""
    role: str  # 'user' или 'assistant'
    content: str
    timestamp: datetime
    sources: Optional[List[str]] = None

class DocumentProcessor:
    """Обработчик документов для RAG"""
    def __init__(self, chunk_size: int = 1000, chunk_overlap: int = 200):
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            separators=["\n\n", "\n", ".", "!", "?"]
        )
        self.encoding = tiktoken.get_encoding("cl100k_base")

    def load_documents(self, directory_path: str) -> List[Dict[str, Any]]:
        """Загрузка документов из директории"""
        documents = []

        # Поддерживаемые форматы
        loaders = {
            '.txt': TextLoader,
            '.md': TextLoader,
            '.pdf': PyPDFLoader
        }

        for ext, loader_class in loaders.items():
            try:
                loader = DirectoryLoader(
                    directory_path,
                    glob=f"**/*{ext}",
                    loader_cls=loader_class,
                    silent_errors=True
                )
                docs = loader.load()
                for doc in docs:
                    # Разбиваем на чанки
                    chunks = self.text_splitter.split_text(doc.page_content)

                    for i, chunk in enumerate(chunks):
                        documents.append({
                            'content': chunk,
                            'source': doc.metadata.get('source', 'unknown'),
                            'chunk_id': i,
                            'token_count': len(self.encoding.encode(chunk))
                        })

            except Exception as e:
                logger.error(f"Ошибка загрузки {ext} файлов: {e}")

        logger.info(f"Загружено {len(documents)} документов")
        return documents

class VectorStore:
    """Векторное хранилище на основе ChromaDB"""

    def __init__(self, collection_name: str = "rag_chatbot",
                 embedding_model: str = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"):
        self.client = chromadb.Client()
        self.collection_name = collection_name
        self.embedding_model = SentenceTransformer(embedding_model)

        try:
            self.collection = self.client.get_collection(collection_name)
            logger.info(f"Найдена существующая коллекция: {collection_name}")
        except:
            self.collection = self.client.create_collection(collection_name)
            logger.info(f"Создана новая коллекция: {collection_name}")

    def add_documents(self, documents: List[Dict[str, Any]]):
        """Добавление документов в векторное хранилище"""
        if not documents:
            return

        # Генерируем эмбеддинги
        texts = [doc['content'] for doc in documents]
        embeddings = self.embedding_model.encode(texts).tolist()

        # Подготавливаем метаданные
        metadatas = []
        ids = []

        for i, doc in enumerate(documents):
            metadata = {
                'source': doc['source'],
                'chunk_id': str(doc['chunk_id']),
                'token_count': str(doc['token_count'])
            }
            metadatas.append(metadata)
            ids.append(f"{doc['source']}_{doc['chunk_id']}")

        # Добавляем в коллекцию
        self.collection.add(
            embeddings=embeddings,
            documents=texts,
            metadatas=metadatas,
            ids=ids
        )

        logger.info(f"Добавлено {len(documents)} документов в векторное хранилище")

    def search(self, query: str, n_results: int = 5) -> List[Dict[str, Any]]:
        """Поиск релевантных документов"""
        
        # Генерируем эмбеддинг для запроса
        query_embedding = self.embedding_model.encode([query]).tolist()
        
        # Выполняем поиск
        results = self.collection.query(
            query_embeddings=[query_embedding],
            n_results=n_results
        )
        
        # Форматируем результаты
        search_results = []
        for i in range(len(results['documents'][0])):
            search_results.append({
                'content': results['documents'][0][i],
                'distance': results['distances'][0][i],
                'metadata': results['metadatas'][0][i]
            })
        
        return search_results

class LLMInterface:
    """Интерфейс для работы с языковой моделью"""
    
    def __init__(self, api_key: str = None, model: str = "gpt-3.5-turbo"):
        self.client = openai.OpenAI(api_key=api_key or os.getenv("OPENAI_API_KEY"))
        self.model = model
        self.max_context_tokens = 4000  # Оставляем место для ответа
    
    def generate_response(self, query: str, context: List[Dict[str, Any]], chat_history: List[ChatMessage] = None) -> Dict[str, Any]:
        """Генерация ответа на основе контекста"""
        
        # Формируем контекст из найденных документов
        context_text = "\n\n".join([
            f"Источник: {doc['metadata'].get('source', 'unknown')}\n{doc['content']}"
            for doc in context
        ])
        
        # Формируем промпт
        system_prompt = f"""Вы - полезный ассистент, который отвечает на вопросы пользователей на основе предоставленного контекста.

        Правила:
        1. Отвечайте только на основе предоставленной информации
        2. Если информации недостаточно, так и скажите
        3. Указывайте источники информации
        4. Отвечайте на русском языке
        5. Будьте точны и полезны

        Контекст:
        {context_text}
        """

        # Подготавливаем историю диалога
        messages = [{"role": "system", "content": system_prompt}]

        if chat_history:
            for msg in chat_history[-5:]:  # Последние 5 сообщений
                messages.append({
                    "role": msg.role,
                    "content": msg.content
                })

        messages.append({"role": "user", "content": query})

        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=messages,
                max_tokens=1000,
                temperature=0.1
            )

            answer = response.choices[0].message.content.strip()

            # Извлекаем источники
            sources = list(set([
                doc['metadata'].get('source', 'unknown')
                for doc in context
            ]))

            return {
                'answer': answer,
                'sources': sources,
                'tokens_used': response.usage.total_tokens if response.usage else 0
            }

        except Exception as e:
            logger.error(f"Ошибка генерации ответа: {e}")
            return {
                'answer': "Извините, произошла ошибка при генерации ответа.",
                'sources': [],
                'tokens_used': 0
            }

class RAGChatbot:
    """Основной класс чат-бота с RAG"""

    def __init__(self, documents_path: str = None, openai_api_key: str = None):
        self.vector_store = VectorStore()
        self.llm = LLMInterface(api_key=openai_api_key)
        self.doc_processor = DocumentProcessor()
        self.chat_history: List[ChatMessage] = []

        # Инициализируем базу знаний
        if documents_path and os.path.exists(documents_path):
            self.initialize_knowledge_base(documents_path)

    def initialize_knowledge_base(self, documents_path: str):
        """Инициализация базы знаний"""
        logger.info("Инициализация базы знаний...")

        # Загружаем и обрабатываем документы
        documents = self.doc_processor.load_documents(documents_path)

        if documents:
            # Добавляем в векторное хранилище
            self.vector_store.add_documents(documents)
            logger.info("База знаний успешно инициализирована")
        else:
            logger.warning("Документы не найдены")

    def chat(self, user_message: str) -> ChatMessage:
        """Основной метод для общения с ботом"""

        # Добавляем сообщение пользователя в историю
        user_msg = ChatMessage(
            role="user",
            content=user_message,
            timestamp=datetime.now()
        )
        self.chat_history.append(user_msg)

        # Ищем релевантные документы
        search_results = self.vector_store.search(user_message, n_results=5)

        # Генерируем ответ
        llm_response = self.llm.generate_response(
            query=user_message,
            context=search_results,
            chat_history=self.chat_history[:-1]  # Исключаем последнее сообщение
        )

        # Создаем ответное сообщение
        assistant_msg = ChatMessage(
            role="assistant",
            content=llm_response['answer'],
            timestamp=datetime.now(),
            sources=llm_response['sources']
        )

        self.chat_history.append(assistant_msg)
        return assistant_msg

    def clear_history(self):
        """Очистка истории диалога"""
        self.chat_history = []

    def get_chat_history(self) -> List[ChatMessage]:
        """Получение истории диалога"""
        return self.chat_history.copy()

# Веб-интерфейс на Streamlit
def create_streamlit_interface():
    """Создание веб-интерфейса"""
    st.set_page_config(
        page_title="RAG Чат-бот",
        page_icon="🤖",
        layout="wide"
    )

    st.title("🤖 Интеллектуальный чат-бот с RAG")
    st.markdown("Задавайте вопросы на основе загруженной базы знаний")

    # Инициализация сессии
    if 'chatbot' not in st.session_state:
        st.session_state.chatbot = None

    # Сайдбар для настроек
    with st.sidebar:
        st.header("⚙️ Настройки")

        # API-ключ
        api_key = st.text_input(
            "OpenAI API Key:",
            type="password",
            help="Введите ваш API-ключ от OpenAI"
        )

        # Путь к документам
        docs_path = st.text_input(
            "Путь к документам:",
            value="./documents",
            help="Укажите путь к папке с документами"
        )

        # Инициализация бота
        if st.button("🤖 Инициализировать бота"):
            if api_key:
                with st.spinner("Инициализация..."):
                    st.session_state.chatbot = RAGChatbot(
                        documents_path=docs_path,
                        openai_api_key=api_key
                    )
                st.success("Бот успешно инициализирован!")
            else:
                st.error("Введите OpenAI API-ключ")

        # Очистка истории
        if st.button("Очистить историю"):
            if st.session_state.chatbot:
                st.session_state.chatbot.clear_history()
                st.rerun()

    # Основной интерфейс чата
    if st.session_state.chatbot:
        # Отображение истории
        history = st.session_state.chatbot.get_chat_history()

        for message in history:
            with st.chat_message(message.role):
                st.write(message.content)

            if message.role == "assistant" and message.sources:
                with st.expander("Источники"):
                    for source in message.sources:
                        st.text(f"• {source}")

        # Поле ввода
        if prompt := st.chat_input("Задайте ваш вопрос…"):
            # Отображаем сообщение пользователя
            with st.chat_message("user"):
                st.write(prompt)

            # Генерируем ответ
            with st.chat_message("assistant"):
                with st.spinner("Обрабатываю запрос…"):
                    response = st.session_state.chatbot.chat(prompt)

                st.write(response.content)

            if response.sources:
                with st.expander("Источники"):
                    for source in response.sources:
                        st.text(f"• {source}")
    else:
        st.info("Настройте и инициализируйте бота в боковой панели")

# Пример использования в командной строке
def demo_usage():
    """Демонстрация использования чат-бота"""
    print("=== Демонстрация RAG чат-бота ===\n")

    # Инициализируем бота
    chatbot = RAGChatbot(
        documents_path="./documents",  # Укажите путь к вашим документам
        openai_api_key="your-openai-api-key"  # Укажите ваш API-ключ
    )

    # Примеры вопросов
    questions = [
        "Что такое искусственный интеллект?",
        "Как работают нейронные сети?",
        "Расскажи о применении машинного обучения"
    ]

    for question in questions:
        print(f"? Вопрос: {question}")
        response = chatbot.chat(question)
        print(f"🤖 Ответ: {response.content}")

        if response.sources:
            print(f"📂 Источники: {', '.join(response.sources)}")

        print("-" * 50)

if __name__ == "__main__":
    # Запуск веб-интерфейса
    create_streamlit_interface()

### Файл зависимостей requirements.txt

In [ ]:
# Основные зависимости для RAG-чат-бота

# Веб-фреймворк
streamlit>=1.28.0

# Работа с векторами и эмбеддингами
chromadb>=0.4.15
sentence-transformers>=2.2.2

# LLM-интеграция
openai>=1.3.0
tiktoken>=0.5.0

# Обработка документов
langchain>=0.1.0
PyPDF2>=3.0.1

# Утилиты
python-dotenv>=1.0.0
requests>=2.31.0

# Дополнительные форматы документов (опционально)
python-docx>=0.8.11
openpyxl>=3.1.0


### Пример файла настроек .env

In [ ]:
# Настройки окружения для RAG-чат-бота

# OpenAI API-ключ
OPENAI_API_KEY=your_openai_api_key_here

# Путь к документам
DOCUMENTS_PATH=./documents

# Настройки модели
EMBEDDING_MODEL=sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
LLM_MODEL=gpt-3.5-turbo

# Настройки чанкинга
CHUNK_SIZE=1000
CHUNK_OVERLAP=200

# Настройки поиска
SEARCH_TOP_K=5



### Инструкции по запуску

In [ ]:
# Установка зависимостей
pip install -r requirements.txt

# Создание папки для документов
mkdir documents

# Запуск веб-интерфейса
streamlit run rag_chatbot.py

# Альтернативно: запуск демо в консоли
python rag_chatbot.py


Это готовый к использованию чат-бот RAG для корпоративных задач. Он умеет обрабатывать разные форматы документов, искать нужную информацию и отвечать на вопросы через удобный веб-интерфейс.

**Основные компоненты системы:**

**DocumentProcessor** — загружает и обрабатывает документы (TXT, PDF, Markdown). Разбивает их на части с перекрытием, чтобы не терялся контекст.

**VectorStore** — хранит и ищет информацию с помощью ChromaDB. Использует многоязычную модель, которая хорошо работает с русским языком.

**LLMInterface** — подключается к OpenAI API. Сам управляет контекстом, вставляет найденные документы в запрос и поддерживает историю диалога.

**RAGChatbot** — объединяет все компоненты. Обрабатывает запросы, ищет документы, генерирует ответы и сохраняет историю.

**Streamlit-интерфейс** — превращает бота в веб-приложение. Можно настраивать через боковую панель и видеть источники ответов.

**Главные преимущества:**

- Модульная архитектура — можно легко заменить любую часть (например, ChromaDB на Pinecone)
- Настройка через переменные окружения без изменения кода
- Обработка ошибок и логирование для стабильной работы
- Подходит как основа для разных корпоративных задач — от техподдержки до обучения

Код демонстрирует современные подходы к созданию RAG-систем и может служить стартовой точкой для более сложных проектов.